In [0]:
storage_account_name = "hkhiarstore"
container_name = "hkhiar-datalake"
storage_account_key = dbutils.secrets.get(scope="azure_access_key",key="storage_key")
azure_key_conf = f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net"


# Définition du chemin d'accès vers la couche Bronze
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
bronze_base_path = base_path + "Bronze/"

try:
    print("Début du chargement des fichiers Parquet depuis la couche Bronze...\n")
    
    # 1. Chargement des Transactions
    df_transactions = (spark.read
                       .format("parquet")
                       .option(azure_key_conf, storage_account_key)
                       .load(f"{bronze_base_path}/transaction/"))
    print("✔ Source 'Transactions' chargée.")

    # 2. Chargement des Produits
    df_products = (spark.read
                   .format("parquet")
                   .option(azure_key_conf, storage_account_key)
                   .load(f"{bronze_base_path}/product/"))
    print("✔ Source 'Products' chargée.")

    # 3. Chargement des Magasins (Stores)
    df_stores = (spark.read
                 .format("parquet")
                 .option(azure_key_conf, storage_account_key)
                 .load(f"{bronze_base_path}/store/"))
    print("✔ Source 'Stores' chargée.")

    # 4. Chargement des Clients (Customers issus de l'API JSON via ADF)
    df_customers = (spark.read
                    .format("parquet")
                    .option(azure_key_conf, storage_account_key)
                    .load(f"{bronze_base_path}/customer/"))
    print("✔ Source 'Customers' chargée.")
    
    print("\n[SUCCÈS] Les 4 DataFrames de la couche Bronze sont prêts à être transformés !")
    
    # Petit affichage de contrôle pour vérifier qu'on a bien tout
    print("\n--- Structure du DataFrame Transactions ---")
    df_transactions.printSchema()
    
    print("--- Aperçu rapide des Clients ---")
    display(df_customers.limit(5))
    
except Exception as e:
    print("\n[ERREUR] Un des dossiers n'a pas pu être chargé. Vérifie le statut du pipeline ADF.")
    print("Détail de l'erreur :", e)

In [0]:
 
print("=== TRANSACTIONS ===")
df_transactions.printSchema()
df_transactions.show(3)
 
print("=== PRODUCTS ===")
df_products.printSchema()
df_products.show(3)
 
print("=== STORES ===")
df_stores.printSchema()
df_stores.show(3)
 
print("=== CUSTOMERS ===")
df_customers.printSchema()
df_customers.show(3)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType, DateType, StringType
 
# Cast des types
df_transactions_clean = df_transactions \
    .withColumn("transaction_id", F.col("transaction_id").cast(IntegerType())) \
    .withColumn("product_id",     F.col("product_id").cast(IntegerType())) \
    .withColumn("store_id",       F.col("store_id").cast(IntegerType())) \
    .withColumn("customer_id",    F.col("customer_id").cast(IntegerType())) \
    .withColumn("quantity",       F.col("quantity").cast(IntegerType())) \
    .withColumn("transaction_date", F.to_date(F.col("transaction_date")))
 
# Suppression des doublons
df_transactions_clean = df_transactions_clean.dropDuplicates(["transaction_id"])
 
# Suppression des lignes avec valeurs nulles critiques
df_transactions_clean = df_transactions_clean.dropna(
    subset=["transaction_id", "product_id", "store_id", "quantity"]
)
 
# Vérification qualité : quantité et prix positifs
df_transactions_clean = df_transactions_clean.filter(
    (F.col("quantity") > 0) 
)
 
print("Transactions après nettoyage :", df_transactions_clean.count())
df_transactions_clean.show(3)
 


In [0]:
df_products_clean = df_products \
    .withColumn("product_id",   F.col("product_id").cast(IntegerType())) \
    .withColumn("price",        F.col("price").cast(FloatType())) \
    .withColumn("product_name", F.trim(F.col("product_name"))) \
    .withColumn("category",     F.trim(F.col("category")))
 
df_products_clean = df_products_clean.dropDuplicates(["product_id"])
df_products_clean = df_products_clean.dropna(subset=["product_id", "product_name"])
 
print("Products après nettoyage :", df_products_clean.count())
df_products_clean.show(3)


In [0]:
df_stores_clean = df_stores \
    .withColumn("store_id",   F.col("store_id").cast(IntegerType())) \
    .withColumn("store_name", F.trim(F.col("store_name"))) \
    .withColumn("location",    F.trim(F.col("location")))
 
df_stores_clean = df_stores_clean.dropDuplicates(["store_id"])
df_stores_clean = df_stores_clean.dropna(subset=["store_id", "store_name"])
 
print("Stores après nettoyage :", df_stores_clean.count())
df_stores_clean.show(3)


In [0]:
df_customers_clean = df_customers \
    .withColumn("customer_id",   F.col("customer_id").cast(IntegerType())) \
    .withColumn("first_name",    F.trim(F.col("first_name"))) \
    .withColumn("last_name",     F.trim(F.col("last_name"))) \
    .withColumn("email",         F.lower(F.trim(F.col("email")))) \
    .dropna(subset=["customer_id"]) \
    .dropDuplicates(["customer_id"])

# 2. Display the results
print("Customers après nettoyage :", df_customers_clean.count())
df_customers_clean.show(3)

In [0]:
df_silver = df_transactions_clean \
    .join(df_products_clean,  on="product_id",  how="left") \
    .join(df_stores_clean,    on="store_id",    how="left") \
    .join(df_customers_clean, on="customer_id", how="left")
 
# Indicateur calculé : total_amount = quantity × price
df_silver = df_silver.withColumn(
    "total_amount",
    F.round(F.col("quantity") * F.col("price"), 2)
)
 
# Sélection des colonnes finales Silver
df_silver = df_silver.select(
    "transaction_id",
    "transaction_date",
    "customer_id",
    F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("customer_name"),
    "email",
    "product_id",
    "product_name",
    "category",
    "store_id",
    "store_name",
    "location",
    "quantity",
    "price",
    "total_amount"
)
 
print("Silver — lignes :", df_silver.count())
df_silver.show(5)
